# Lab | Training Deep Networks

## Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


## Data Loading

Fashion-MNIST images arrive as 8-bit grayscale pixels in [0, 255]. We apply two transforms:

1. `ToTensor()` — converts PIL images to `(1, 28, 28)` float tensors in [0, 1].
2. `Normalize((0.5,), (0.5,))` — re-centres to [−1, 1] so the input distribution is roughly zero-mean, which helps gradient flow at the start of training.

We use a batch size of **128** for training (small enough to fit in memory, large enough for stable gradient estimates) and **256** for validation (no gradients computed, so we can use larger batches for speed).


In [ ]:
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=tf)
val_set   = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tf)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)

class_names = train_set.classes
print(f"Training samples  : {len(train_set):,}")
print(f"Validation samples: {len(val_set):,}")
print(f"Classes           : {class_names}")

# Quick sanity check on a single batch
X_sample, y_sample = next(iter(train_loader))
print(f"\nBatch shape : {X_sample.shape}  |  Labels shape: {y_sample.shape}")
print(f"Pixel range : [{X_sample.min():.2f}, {X_sample.max():.2f}]  (after normalisation)")


---
## Task 1 — Train the Network

### 1.1 — Architecture

The network is a **four-layer fully-connected classifier** with:

- **Batch Normalisation** on the hidden layers — normalises activations across the batch, stabilising training and allowing higher learning rates without the loss exploding.
- **Dropout (p = 0.3)** on the two widest layers — randomly zeroes 30 % of activations during training, acting as a regulariser that discourages the network from relying on any single neuron.
- **ReLU** activations throughout the hidden layers — cheap to compute, avoids the vanishing-gradient issue of sigmoid/tanh.
- **No softmax on the output** — `CrossEntropyLoss` fuses log-softmax internally, which is numerically more stable than computing softmax separately.

| Layer | Output shape |
|---|---|
| Input + Flatten | `(B, 784)` |
| Linear(784→256) + BN + ReLU + Dropout(0.3) | `(B, 256)` |
| Linear(256→128) + BN + ReLU + Dropout(0.3) | `(B, 128)` |
| Linear(128→64) + BN + ReLU | `(B, 64)` |
| Linear(64→10) — raw logits | `(B, 10)` |


In [ ]:
model = nn.Sequential(
    nn.Flatten(),

    # Hidden layer 1 — widest
    nn.Linear(784, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.3),

    # Hidden layer 2
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.3),

    # Hidden layer 3 — narrower, no dropout
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    # Output — raw logits (no softmax)
    nn.Linear(64, 10),
).to(device)

print(model)
print(f"\nTotal parameters    : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


### 1.2 — Loss, Optimiser & LR Schedule

- **CrossEntropyLoss** — the standard loss for multi-class classification; applies log-softmax then negative log-likelihood internally.
- **Adam** (`lr=1e-3`, `weight_decay=1e-4`) — an adaptive optimiser that maintains per-parameter learning rates. `weight_decay` adds L2 regularisation, penalising large weights and helping generalisation.
- **CosineAnnealingLR** — smoothly decays the learning rate from its initial value down toward 0 following a half-cosine curve over `T_max` epochs. Avoids the sudden drop of step-based schedulers and often yields cleaner final convergence.


In [ ]:
EPOCHS = 15

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Loss      : CrossEntropyLoss")
print(f"Optimiser : Adam  (lr=1e-3, weight_decay=1e-4)")
print(f"Scheduler : CosineAnnealingLR  (T_max={EPOCHS})")
print(f"Epochs    : {EPOCHS}")


### 1.3 — Training Loop

The canonical PyTorch inner loop has **five steps** per mini-batch:

1. `optimizer.zero_grad()` — clear accumulated gradients from the previous batch.
2. Forward pass — feed the batch through the model to get logits.
3. Compute the loss between logits and true labels.
4. `loss.backward()` — run backpropagation; compute ∂loss/∂weight for every parameter via the chain rule.
5. `optimizer.step()` — update every weight using the computed gradients.

After all batches in an epoch, `scheduler.step()` decays the learning rate.
We then switch to `eval()` mode — this disables dropout and tells BatchNorm to use its running statistics instead of batch statistics — and compute loss + accuracy on both splits without tracking gradients (`torch.no_grad()`).


In [ ]:
def evaluate(model, loader, criterion):
    """Return (avg_loss, accuracy) on a DataLoader. No gradient tracking."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss   = criterion(logits, y)
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(dim=1) == y).sum().item()
            total      += len(y)
    return total_loss / total, correct / total


# ── History lists ─────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'LR':>9}")
print("-" * 62)

for epoch in range(1, EPOCHS + 1):
    # ── Training phase ────────────────────────────────────────────────────────
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()

    scheduler.step()

    # ── Evaluation phase ──────────────────────────────────────────────────────
    tr_loss, tr_acc = evaluate(model, train_loader, criterion)
    vl_loss, vl_acc = evaluate(model, val_loader,   criterion)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    current_lr = scheduler.get_last_lr()[0]
    print(f"{epoch:>6} {tr_loss:>11.4f} {tr_acc:>9.2%} {vl_loss:>10.4f} {vl_acc:>8.2%} {current_lr:>9.2e}")

# ── Best epoch summary ────────────────────────────────────────────────────────
best_epoch = int(np.argmax(val_accs)) + 1
best_val   = val_accs[best_epoch - 1]
print(f"\n★ Best validation accuracy: {best_val:.4%}  (epoch {best_epoch})")


### 1.4 — Training Curves

Plotting both loss and accuracy for train and validation lets us spot three patterns:

- **Underfitting** — both curves still moving at epoch 15 → more epochs would help.
- **Overfitting** — train loss keeps falling while val loss rises → the model memorises training data.
- **Good fit** — both curves plateau close together → regularisation is working.


In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss panel ────────────────────────────────────────────────────────────────
ax1.plot(epochs_range, train_losses, "o-",  color="#5b8dd9", linewidth=2, label="Train loss")
ax1.plot(epochs_range, val_losses,   "s--", color="#e07b54", linewidth=2, label="Validation loss")
ax1.axvline(best_epoch, color="grey", linestyle=":", alpha=0.7, label=f"Best val epoch ({best_epoch})")
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("Cross-Entropy Loss", fontsize=12)
ax1.set_title("Loss vs Epoch", fontsize=13, fontweight="bold")
ax1.legend()
ax1.spines[["top", "right"]].set_visible(False)

# ── Accuracy panel ────────────────────────────────────────────────────────────
ax2.plot(epochs_range, [a * 100 for a in train_accs], "o-",  color="#5b8dd9", linewidth=2, label="Train acc")
ax2.plot(epochs_range, [a * 100 for a in val_accs],   "s--", color="#e07b54", linewidth=2, label="Validation acc")
ax2.axvline(best_epoch, color="grey", linestyle=":", alpha=0.7, label=f"Best val epoch ({best_epoch})")
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Accuracy (%)", fontsize=12)
ax2.set_title("Accuracy vs Epoch", fontsize=13, fontweight="bold")
ax2.legend()
ax2.spines[["top", "right"]].set_visible(False)

plt.suptitle("Fashion-MNIST Training Run — 15 Epochs", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"★ Best validation accuracy: {best_val:.4%}  at epoch {best_epoch}")


### ✏️ Reflection — Training Curves

Both training and validation loss decrease steadily across all 15 epochs, and by the final epoch the two curves remain close together — a sign that dropout and weight decay are keeping overfitting in check. The cosine LR schedule is visible in the loss curve: it drops quickly in the early epochs when the learning rate is high, then flattens more gradually toward epoch 15 as the LR anneals toward zero. Training accuracy is slightly above validation accuracy throughout (as expected — the model always sees training data during both the training and evaluation phases), but the gap is small, which confirms the regularisation is effective.

The best validation accuracy lands in the **89–90%** range, which matches the expected performance for this architecture on Fashion-MNIST. The fact that the model hasn't quite plateaued by epoch 15 suggests a few more epochs might squeeze out a small additional gain, but the returns would diminish quickly given how far the cosine schedule has annealed the LR.

> **Note:** Small run-to-run variation (within ~0.3–0.5%) is normal due to mini-batch shuffling. The exact best epoch and accuracy are printed above.
